**Getting the dataset using an URL from the huggingface**

In [13]:
import pandas as pd
url = "https://huggingface.co/datasets/JEFFREY-VERDIERE/Creditcard/resolve/feac26da102fcef433202e29a528e3bfb6a62a04/creditcard.csv"
df = pd.read_csv(url)
print(df.shape)

(284807, 31)


In [14]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Checking the number of frauds
num_frauds = int(df["Class"].sum())
print(df.shape, "| fraud cases:", num_frauds)

scaler = StandardScaler()
df["Amount"] = scaler.fit_transform(df[["Amount"]])
df["Time"] = scaler.fit_transform(df[["Time"]])

# features and labels
X = df.drop("Class", axis=1).values
y = df["Class"].values

X_normal = X[y == 0]
X_train, X_test_normal = train_test_split(X_normal, test_size=0.2, random_state=42)
X_fraud = X[y == 1]

X_test = np.vstack([X_test_normal, X_fraud])
y_test = np.hstack([np.zeros(len(X_test_normal)), np.ones(len(X_fraud))])
print("train shape:", X_train.shape, "| test shape:", X_test.shape, "| frauds in test:", int(y_test.sum()))

(284807, 31) | fraud cases: 492
train shape: (227452, 30) | test shape: (57355, 30) | frauds in test: 492


# Model A: Autoencoder (AE)

In [15]:
import tensorflow as tf
from tensorflow.keras import layers, Model

dim = X_train.shape[1]

ae = tf.keras.Sequential([
    layers.Input(shape=(dim,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(dim, activation="linear"),
])
ae.compile(optimizer="adam", loss="mse")

# training AE for 10 epochs
ae.fit(X_train, X_train, epochs=10, batch_size=256, validation_split=0.1, verbose=1)

Epoch 1/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.6898 - val_loss: 0.5037
Epoch 2/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.4285 - val_loss: 0.4009
Epoch 3/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3751 - val_loss: 0.3688
Epoch 4/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3499 - val_loss: 0.3460
Epoch 5/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3302 - val_loss: 0.3279
Epoch 6/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3159 - val_loss: 0.3169
Epoch 7/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.3062 - val_loss: 0.3068
Epoch 8/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2997 - val_loss: 0.3030
Epoch 9/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2949 - val_loss: 0.2965
Epoch 10/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2911 - val_loss: 0.2930


In [16]:
# Evaluating AE on test set
from sklearn.metrics import roc_auc_score

reconstructed_ae = ae.predict(X_test)
ae_reconstruction_error = np.mean((X_test - reconstructed_ae) ** 2, axis=1)   # reconstruction error per transaction
ae_auc_score = roc_auc_score(y_test, ae_reconstruction_error)            # higher error should mean fraud
print("Autoencoder AUC:", round(ae_auc_score, 3))

1793/1793 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Autoencoder AUC: 0.968


# Model B: Variational Autoencoder (VAE)

In [17]:
# Same X_train, X_test and y_test used from above

input_dim = X_train.shape[1]
latent_size = 4 # keeping the latent space small

# Custom sampling layer for the reparameterization trick (based on the keras VAE example)
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# --- ENCODER ---
encoder_inputs = layers.Input(shape=(input_dim,))
x_enc = layers.Dense(16, activation="relu")(encoder_inputs)
x_enc = layers.Dense(8, activation="relu")(x_enc)

z_mean = layers.Dense(latent_size)(x_enc)
z_log_var = layers.Dense(latent_size)(x_enc)

# Get the latent vector using the custom sampling layer
z_sample = Sampling()([z_mean, z_log_var])

encoder = Model(encoder_inputs, [z_mean, z_log_var, z_sample], name="encoder")

# --- DECODER ---
decoder_inputs = layers.Input(shape=(latent_size,))
x_dec = layers.Dense(8, activation="relu")(decoder_inputs)
x_dec = layers.Dense(16, activation="relu")(x_dec)
decoder_outputs = layers.Dense(input_dim, activation="linear")(x_dec)

decoder = Model(decoder_inputs, decoder_outputs, name="decoder")

# VAE class wrapper with custom training step to handle the KL divergence loss properly
class VariationalAutoencoder(Model):
    def __init__(self, enc, dec, **kwargs):
        super().__init__(**kwargs)
        self.encoder = enc
        self.decoder = dec

    def train_step(self, data):
        # handling data formatting quirk in tf datasets
        if isinstance(data, tuple):
            x = data[0]
        else:
            x = data

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x)
            reconstruction = self.decoder(z)

            # Reconstruction loss
            recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(x - reconstruction), axis=1))

            # KL divergence loss
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
            )

            total_loss = recon_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        return {
            "loss": total_loss,
            "recon_loss": recon_loss,
            "kl_loss": kl_loss,
        }

# Instantiate and compile the VAE
vae_model = VariationalAutoencoder(encoder, decoder)
vae_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

print("Training the VAE now...")
vae_model.fit(X_train, epochs=10, batch_size=256, verbose=1)

Training the VAE now...
Epoch 1/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - kl_loss: 2.0456 - loss: 31.4706 - recon_loss: 29.4250
Epoch 2/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - kl_loss: 2.7960 - loss: 21.6457 - recon_loss: 18.8497
Epoch 3/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - kl_loss: 3.1148 - loss: 21.2437 - recon_loss: 18.1289
Epoch 4/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - kl_loss: 3.5045 - loss: 19.6975 - recon_loss: 16.1931
Epoch 5/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - kl_loss: 4.0729 - loss: 21.9788 - recon_loss: 17.9059
Epoch 6/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - kl_loss: 3.4072 - loss: 18.3090 - recon_loss: 14.9018
Epoch 7/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - kl_loss: 3.4777 - loss: 16.9492 - recon_loss: 13.4715
Epoch 8/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - kl_loss: 3.5009 - loss: 16.4925 - recon_loss: 12.9916
Epoch 9/10
889/889 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - kl_loss: 3.5381 - loss: 18.0424 - recon_loss: 14.5043
Epoc

In [18]:
# Evaluate VAE on test set
print("Running predictions on test set...")
mu, log_v, latent_z = encoder.predict(X_test)
reconstructed_vae = decoder.predict(latent_z)

# Calculate error (MSE per sample)
vae_reconstruction_error = np.mean(np.power(X_test - reconstructed_vae, 2), axis=1)
vae_auc_score = roc_auc_score(y_test, vae_reconstruction_error)

print("VAE AUC:", round(vae_auc_score, 3))
# Printing the old one too for comparison
try:
    print("AE AUC was:", round(ae_auc_score, 3))
except NameError:
    print("AE AUC variable not found")

Running predictions on test set...
1793/1793 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
1793/1793 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
VAE AUC: 0.957
AE AUC was: 0.968
